# 다중분류 NN
- 펭귄 데이터 셋



In [ ]:
!pip install ipython-autotime
%load_ext autotime

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras

SEED = 42

## 1. 데이터 준비

In [ ]:
data_url = 'https://raw.githubusercontent.com/20161609/data_box/refs/heads/main/penguins.csv'
df = pd.read_csv(data_url)
df.shape

In [ ]:
file_path = 'penguins.csv'
penguins = pd.read_csv(file_path)
penguins.shape # -> 펭귄을 3종류로 분리할 것

In [ ]:
df = penguins.copy()
df.head()

In [ ]:
col_dict = {col: col.lower().replace(' ', '_').replace('(', '_').replace(')', '') for col in df.columns}
df.rename(columns=col_dict, inplace=True)
df.head()

### 범주형 변수

In [ ]:
cols_cat = df.select_dtypes(include='object')
cols_cat

In [ ]:
s =  cols_cat['species'].value_counts()

In [ ]:
cols_cat['island'].value_counts()

In [ ]:
cols_cat['sex'].value_counts()

In [ ]:
cols_cat.loc[cols_cat['sex']=='.','sex'] = 'FEMALE'
cols_cat['sex'].value_counts()

In [ ]:
df.groupby('island')['species'].value_counts()

### 연속형 변수

In [ ]:
cols_num = df.select_dtypes(include='number')
cols_num

In [ ]:
cols_num.hist(figsize=(10,8))

## 이상치

In [ ]:
# 박스플롯
fig, axes = plt.subplots(2, 2, figsize=(10,8))
axes = axes.flatten()

for i, col in enumerate(cols_num.columns):
  sns.boxplot(y=col, data=cols_num, hue=df['species'],ax=axes[i])
  axes[i].set_xlabel(None)
  axes[i].set_ylabel(None)
  axes[i].set_title(col)

In [ ]:
# 데이터 분리
# 결측치
# 인코딩
# 학습

## 2. 트레인, 테스트 분리

In [ ]:
from sklearn.model_selection import train_test_split
SEED = 42
train, test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['species'])

train.shape, test.shape

### 결측치처리

In [ ]:
train.isna().sum()
train

In [ ]:
sns.heatmap(train.isna()) # 흰줄이 결측치

In [ ]:
train = train.dropna(axis=0)
train

### X, y 분리

In [ ]:
X_train = train.drop('species', axis=1)
y_train = train['species']

X_train.shape, y_train.shape

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder()

X_train[['island','sex']] = oe.fit_transform(X_train[['island','sex']])
X_train.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_e = le.fit_transform(y_train)
y_train_e

In [ ]:
from keras.utils import to_categorical

y_train_o = to_categorical(y_train_e)
y_train_o

## 스케일링

In [ ]:
from sklearn.preprocessing import RobustScaler

rs = RobustScaler()
X_train_s = rs.fit_transform(X_train)
X_train_s

# 3. 모델 학습

In [ ]:
print(X_train_s.shape, y_train.shape)
print(type(X_train_s), type(y_train_o))


In [ ]:
from keras import layers
model = keras.Sequential([
    layers.Dense(16, activation='relu', input_shape=(6,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(3, activation='softmax') # 다중 분류는 시그모이드로 가라 -> 확률로 나옴
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
EPOCH = 100
BATCH_SIZE = 16
history = model.fit(
    X_train_s, y_train_o,
    epochs=EPOCH,
    batch_size=BATCH_SIZE,
    validation_split=0.2
)

In [ ]:
def plot_history(history):
    hist = pd.DataFrame(history.history)
    hist['epoch'] = history.epoch

    plt.figure(figsize=(16, 8))
    plt.subplot(1, 2, 1)
    plt.xlabel('epochs')
    plt.ylabel('loss')
    plt.plot(hist['epoch'], hist['loss'], label='train loss')
    plt.plot(hist['epoch'], hist['val_loss'], label='val loss')
    plt.title('Loss Curve')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.xlabel('epochs')
    plt.ylabel('accuracy')
    plt.plot(hist['epoch'], hist['accuracy'], label='train accuracy')
    plt.plot(hist['epoch'], hist['val_accuracy'], label='val accuracy')
    plt.title('Accuracy Curve')
    plt.legend()
    plt.show()

In [ ]:
plot_history(history)

# 예측

In [ ]:
# 평가 확인 (테스트 데이터 전처리를 하기 귀찮아서 ㅇㅇ)
X_test_s = X_train_s.copy()
y_test_e = y_train_e.copy()


In [ ]:
y_pred = model.predict(X_test_s)
y_pred

In [ ]:
import numpy as np

y_pred = np.argmax(y_pred, axis=1)
y_pred

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.metrics import confusion_matrix

def print_metrics(y_true, y_pred, ave='binary'):
  print('accuracy:', accuracy_score(y_test_e, y_pred))
  print('recall:', recall_score(y_test_e, y_pred, average=ave))
  print('precision:', precision_score(y_test_e, y_pred, average=ave))
  print('f1 :', f1_score(y_test_e, y_pred, average=ave))

  clm = confusion_matrix(y_test_e, y_pred)
  s = sns.heatmap(clm, annot=True, cmap='Blues', fmt='d', cbar=False)
  s.set(xlabel='Predicted', ylabel='Actual')

In [ ]:
print_metrics(y_test_e, y_pred, ave='macro')

### 트리 알고리즘

In [ ]:
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(random_state=SEED, max_depth=3)
clf.fit(X_train_s, y_train_e)


In [ ]:
from sklearn.tree import plot_tree

plot_tree(clf, filled=True, feature_names=X_train.columns)
plt.show();

In [ ]:
# 테스트 데이터 전처리
X_test = test.drop('species', axis=1)
y_test = test['species']

X_test[['island','sex']] = oe.transform(X_test[['island','sex']])
y_test_e = le.transform(y_test)

# 평가
X_test_s = rs.transform(X_test)

## 랜덤 포레스트

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf_r = RandomForestClassifier(n_estimators=100, random_state=SEED)
clf_r.fit(X_train_s, y_train_e)


In [ ]:
X_train.columns

In [ ]:
clf.feature_importances_

In [ ]:
y_pred = clf_r.predict(X_test_s)
y_pred

In [ ]:
from sklearn.metrics import confusion_matrix

cfm = confusion_matrix(y_test_e, y_pred)
s = sns.heatmap(cfm, annot=True, cmap='Blues', fmt='d', cbar=False)
s.set(xlabel='Prediction', ylabel='Actual')

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score,f1_score

print('accuracy:', accuracy_score(y_test_e, y_pred))
print('recall:', recall_score(y_test_e, y_pred, average='macro'))
print('precision:', precision_score(y_test_e, y_pred, average='macro'))
print('f1 :', f1_score(y_test_e, y_pred, average='macro'))

plt.show()

In [ ]:
path = '/content/bank.csv'

df2 = pd.read_csv(path)
df2.columns